# Практическая работа, модуль 18, тема 5 
# NLP

#### Цели работы

* Освоить интерфейс TfidfVectorizer.
* Закрепить навыки токенизации текста.
* Закрепить навыки удаления стоп-слов и лемматизации текста.
* Научиться векторизовать текст.

Сдавать на провекру выполненную работу не нужно.  

К занятию приложен файл `vacansies.csv`, который содержит описание вакансий одной большой IT-компании. Скачайте его себе на компьютер.

### Что нужно сделать
В файле `vacansies.csv` — сотни разных вакансий, некоторые из них — похожи. Найдите вакансию с наибольшим числом похожих вакансий. Считайте, что вакансии похожи, если косинусное расстояние между векторами, которые представлют их тексты, меньше `0.5`.

##### Шаг 1
Считайте файл `vacansies.csv` в Pandas-dataframe `df`.

##### Шаг 2
Напишите функцию `preprocess`, которая:
1. Принимает текст с описанием вакансии в качестве аргумента.
1. Токенизирует его (в данном случае токен — это отдельное слово). Обратите внимание, что описания вакансий организованы сложнее, чем тексты, с которыми вы работали в модуле. В текстах вакансий есть знаки препинания, переносы строк, emoji и прочее, поэтому просто использовать функцию [split](https://docs.python.org/3/library/stdtypes.html#str.split) не получится. Рекомендуем взять из NLTK [RegexpTokenizer](https://www.nltk.org/api/nltk.tokenize.RegexpTokenizer.html), который токенизирует текст с помощью регулярного выражения: всё, что ему удовлетворяет, считается токеном.
1. Удаляет из множества токенов (слов) стоп-слова.
1. Приводит каждый токен (слово) к нормальной форме (лемме). В модуле вы использовали стеммер Портера из NLTK, теперь попробуйте для разнообразия [MorphAnalyzer](https://pymorphy2.readthedocs.io/en/stable/misc/api_reference.html#pymorphy2.analyzer.MorphAnalyzer) из pymorphy2.
1. Возвращает предобработанный текст, который состоит из токенов (слов) в нормальной форме и не содержит стоп-слов.

##### Шаг 3
Создайте экземпляр [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), передайте ему вашу функцию `preprocess`. Нормализацию не используйте. 
##### Шаг 4
Пропустите датафрейм `df` с текстами вакансий через TfidfVectorizer, а затем, как мы делали в модуле, создайте датафрем `result` на основе того, что вернёт векторизатор. Если всё сделано правильно, столбцы этого датафрейма — это слова, строки — документы, значения в ячейках — метрика TF-IDF для данного слова в данном документе.
##### Шаг 5
Рассчитайте косинусное расстояние для всех векторов корпуса попарно. Используйте функцию [cosine_distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_distances.html). Результат сохраните в датафрейм `dist`. 
##### Шаг 6
Используя функции pandas, найдите вакансию с максимальным количеством похожих на неё вакансий. Подсказка: вектор косинусных расстояний у такой вакансии должен иметь большее всего элементов, значения которых меньше `0.5`.

In [ ]:
import pandas as pd

#
# Ваш код здесь.
#

## Тема 5. Решение


In [ ]:
import pandas as pd

### ШАГ 1
df = pd.read_csv('data/skillbox/ml_jun18/vacancies.csv')

df.head()

,text
0,Старший Java-разработчик в Музыку🎧\n\nВас ждет...
1,Python-разработчик в Яндекс.Лавку🍔\n\nЯндекс.Л...
2,Фронтенд-разработчик в Вертикали🏠\n\nВертикали...
3,iOS-разработчик в Вертикали (Буткемп)🍏\n\nВерт...
4,Старший разработчик в группу разработки бессер...


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
import pymorphy2

tokenizer = RegexpTokenizer('\w+')
russian_stopwords = stopwords.words('russian')
morph = pymorphy2.MorphAnalyzer()

### ШАГ 2
def preprocess(text):
    stemmed_words = []
    for word in tokenizer.tokenize(text):
        word = word.lower()
        if word not in russian_stopwords:
            stemmed_words.append(morph.parse(word)[0].normal_form)
    return ' '.join(stemmed_words)

### ШАГ3
vectorizer = TfidfVectorizer(
    preprocessor=preprocess,
    norm=None
)

### ШАГ 4
tfidf_matrix = vectorizer.fit_transform(df['text'])

result = pd.DataFrame(
    data=tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

result

,000,06jjq6ru3cyxcp,0a0a1tvvgd6qw,0h4wwufcxmp3u,0ihk9s7cecxzn,0l4_0nwk3uwaku,0lrks90zuaid4,10,100,1000,...,юридический,явление,являться,ядро,язык,языковой,яндекс,яндекс360,яп,ящик
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.635329,0.0,2.741850,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,2.741850,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,1.370925,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.635329,0.0,1.370925,0.0,0.0,0.0
621,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,1.370925,0.0,0.0,0.0
622,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,1.370925,0.0,0.0,0.0
623,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,1.370925,0.0,0.0,0.0


In [ ]:
from sklearn.metrics.pairwise import cosine_distances

### ШАГ 5
distances = cosine_distances(result)

dist = pd.DataFrame(distances)

dist.head()

,0,1,2,3,4,5,6,7,8,9,...,615,616,617,618,619,620,621,622,623,624
0,0.000000,0.867277,0.892791,0.898116,0.890252,0.924523,0.863116,0.004905,0.937734,0.744903,...,0.969723,0.962532,0.948985,0.937083,0.958178,0.852990,0.901102,0.950424,0.940666,0.940666
1,0.867277,0.000000,0.850028,0.883378,0.932432,0.910260,0.872761,0.866623,0.851844,0.941676,...,0.942066,0.930739,0.945752,0.966840,0.940882,0.868696,0.930068,0.851514,0.873019,0.873019
2,0.892791,0.850028,0.000000,0.705472,0.956243,0.895629,0.910963,0.910157,0.919442,0.954458,...,0.782189,0.694877,0.892991,0.798895,0.957063,0.945965,0.944117,0.950632,0.916042,0.916042
3,0.898116,0.883378,0.705472,0.000000,0.953029,0.922851,0.918541,0.897613,0.956812,0.953168,...,0.784778,0.886813,0.906069,0.974369,0.950082,0.937717,0.952347,0.966035,0.972255,0.972255
4,0.890252,0.932432,0.956243,0.953029,0.000000,0.911678,0.921108,0.889711,0.939244,0.929367,...,0.976503,0.975582,0.965005,0.937016,0.966852,0.897681,0.917412,0.966232,0.971561,0.971561


In [ ]:
### ШАГ 6
dist.apply(lambda x: x[x < 0.5].count()).idxmax()

df.iloc[143]

text    Разработчик-аналитик машинного обучения в Еду🍎...
Name: 143, dtype: object

In [ ]:
### Дополнительно: все похожие вакансии.
dist.iloc[143][dist.iloc[143]<0.5]

143    0.000000
221    0.197556
283    0.371167
293    0.415468
359    0.427564
365    0.404273
485    0.414981
486    0.414981
Name: 143, dtype: float64

In [ ]:
### Например
df.iloc[221]

text    Аналитик ML в Еду🍇\n\nЯндекс.Еда — быстро раст...
Name: 221, dtype: object